# 8. Pitfalls: instability, inference and leakage

Regularised regression is easy to fit and easy to misread. This notebook covers the mistakes that
come up in practice and in interviews.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(42)

## 1. Lasso selection is unstable with correlated predictors
Refit on bootstrap samples and count how often each variable is selected.

In [2]:
# The project's real data: the horizon-2 design (nowcast made at the end of the quarter)
from nowcast.features import build_all, main_spec
from nowcast.models import Standardiser
B = build_all([2])
X = B["X"][2]
X = X[[c for c in X.columns if main_spec(c)]]       # real-time output + other indicators
y = B["first"].reindex(X.index)                      # target: ONS's first published estimate
keep = y.notna() & ~X.index.isin(pd.period_range("2020Q1", "2021Q2", freq="Q"))
X, y = X[keep], y[keep]
names = list(X.columns)
print(X.shape, "quarters x features")

(108, 236) quarters x features


In [3]:
from sklearn.linear_model import Lasso
st = Standardiser().fit(X.values); Z = st.transform(X.values); yy = y.values
counts = np.zeros(Z.shape[1]); B_ = 100
for b in range(B_):
    idx = rng.choice(len(yy), len(yy), replace=True)
    counts += Lasso(alpha=0.03, max_iter=20000).fit(Z[idx], yy[idx]).coef_ != 0
freq = pd.Series(counts / B_, index=names).sort_values(ascending=False)
print("Bootstrap selection frequency (lambda fixed at 0.03):")
print(freq.head(15).round(2).to_string())
print(f"\nfeatures selected in >90% of resamples: {(freq > 0.9).sum()};"
      f" in 10-90%: {((freq > 0.1) & (freq <= 0.9)).sum()}")

Bootstrap selection frequency (lambda fixed at 0.03):
RT_IOS_l2     0.89
D7BT_l3       0.85
RT_IOP_l0     0.81
GDP_q1        0.79
J5DZ_l0       0.74
RT_IOS_l4     0.73
MGRZ_l0       0.71
XUMAUSS_l0    0.71
RT_IOM_l2     0.69
RT_IOP_l1     0.66
LPMAUYN_l0    0.64
AP2Y_l3       0.60
J5EK_l1       0.55
LPMAUYN_l3    0.52
IUMBV34_l0    0.51

features selected in >90% of resamples: 0; in 10-90%: 73


No single feature is selected in more than 90% of resamples. The real-time output lags come
closest (70-90%), and dozens of features flicker in and out. **Stability selection**
(Meinshausen and Bühlmann, 2010) formalises this: keep variables selected in a high share of
subsamples. Report selection frequencies, not a single selected set.

## 2. A selected variable is not "significant"
After the lasso picks variables, refitting OLS on them and reading the usual p-values is invalid:
the same data chose the variables, so the p-values are far too small. Demonstration with pure noise:

In [4]:
import statsmodels.api as sm
from sklearn.linear_model import LassoCV
pvals = []
for rep in range(200):
    Xn = rng.normal(size=(100, 50)); yn = rng.normal(size=100)          # no true signal at all
    sel = np.where(LassoCV(cv=5, alphas=30).fit(Xn, yn).coef_ != 0)[0]
    if len(sel):
        pvals += list(sm.OLS(yn, sm.add_constant(Xn[:, sel])).fit().pvalues[1:])
pvals = np.array(pvals)
print(f"share of naive post-selection p-values < 0.05 under pure noise: {np.mean(pvals < 0.05):.0%} (should be 5%)")

share of naive post-selection p-values < 0.05 under pure noise: 51% (should be 5%)


Valid alternatives: sample splitting (select on one half, test on the other), selective inference
(Lee et al., 2016), or the debiased/desparsified lasso. For forecasting, the honest test is
out-of-sample accuracy, which is what the main project reports.

## 3. Leakage through preprocessing
Standardising (or imputing, or choosing variables) using the whole sample, then backtesting, lets
future information in. In this project the `Standardiser` is fitted inside every backtest origin
on that origin's training rows only.

## 4. Reading coefficients under correlation
A coefficient is the effect of a feature *holding the others fixed*. With 236 correlated features
that is rarely meaningful. Interpret groups (selection frequency by indicator group) and
contributions to the forecast, not single coefficients.

## 5. Look-ahead bias in the data themselves
Macroeconomic data are revised. Evaluating against today's GDP figures, or using today's vintage of
the indicators, flatters a model. Today's monthly GDP series, for example, aggregates almost
exactly to today's quarterly GDP; using it to "nowcast" past quarters borrows years of later
revisions. This project evaluates against the **first published** estimate from the ONS real-time
database, reads the output indices from the ONS **revisions triangles** (each month's figure as it
was published at the time), and measures the look-ahead bias directly by re-running the backtest
with latest-vintage output data.

**Key takeaways**
* Report selection frequencies, not a single selected list.
* No naive p-values after selection.
* Fit every preprocessing step inside the backtest loop.
* Interpret groups and contributions, not individual coefficients.